# 02 · SFT from scratch, with loss masking

Build supervised fine-tuning by hand on a model small enough to train in seconds: pretrain a tiny GPT on plain
text, then teach it a chat format — building the `labels` with `-100` on the prompt so that only the assistant's
tokens are learned. Afterwards you can explain (and debug) loss masking, the EOS token, and packing vs padding.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min, no downloads · **Needs:**
[01 · When to fine-tune and data prep](01_when_to_fine_tune_and_data_prep.ipynb) (chat templates),
[Transformer block and tiny GPT](../07_genai_foundations/04_transformer_block_and_tiny_gpt.ipynb) (the model)

## Why this matters in interviews

- "Your fine-tuned model is worse than the base — debug it" (`tn32`). The two silent, common causes are the chat
  template and the loss mask. Naming them first shows you have debugged a fine-tune, not just read about one.
- Every SFT library has a switch for "train on completions only". You should know what it does underneath:
  `labels = -100` on every prompt position, and the loss averaged over the rest.
- EOS and packing questions separate people who ran training from people who ran `trainer.train()`.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn01` | How is an LLM trained, start to finish? |
| `tn02` | Explain next-token prediction as a training objective. Why is it enough? |
| `fo09` | What is the difference between a base model, an instruct model and a chat model? |
| `tn24` | What is a chat template and why does it matter when self-hosting? (loss masking) |
| `tn32` | Your fine-tuned model performs worse than the base model. How do you debug it? |
| `tn18` | What does the training data pipeline look like for a pretrained model? (packing) |
| `tn22` | What is the difference between an epoch, a step and a batch in LLM training? |
| `tn14` | What hyperparameters matter (learning rate, epochs, batch size)? |

In [ ]:
import copy, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch

torch.set_num_threads(1)          # a tiny model: one thread is fast enough and gives identical numbers everywhere
random.seed(0); np.random.seed(0); torch.manual_seed(0)

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title: ax.set_title(title)
    plt.show()

draw_pipeline(["pretrain\n(plain text)", "SFT\n(prompt → answer)", "preference tuning\n(chosen vs rejected)"],
              title="How an LLM is trained: this notebook does stages 1 and 2 in miniature")

## 1. A tiny world, a tiny tokenizer, a tiny GPT

**In plain English:** pretraining is where knowledge comes from — the model reads text and learns to predict the
next word. Our "internet" is a hundred sentences about capital cities and animals. SFT comes afterwards and teaches
a *format*: a user asks, the assistant answers, then stops.

The tokenizer is word-level, with five special tokens that play the roles real models give to `<|endoftext|>`,
`<|im_start|>user`, `<|im_start|>assistant` and `<|im_end|>`:

| our token | real-model analogue | job |
|---|---|---|
| `<pad>` | pad | fills batches to equal length; never trained on |
| `<eot>` | `<|endoftext|>` | separates pretraining documents |
| `<user>` / `<assistant>` | `<|im_start|>user` / `<|im_start|>assistant` | who is speaking |
| `<eos>` | `<|im_end|>` | end of the assistant's turn: the model must learn to emit it to *stop* |

In [ ]:
countries = [("france", "paris"), ("japan", "tokyo"), ("italy", "rome"), ("spain", "madrid"), ("egypt", "cairo"),
             ("kenya", "nairobi"), ("peru", "lima"), ("chile", "santiago"), ("canada", "ottawa"), ("india", "delhi"),
             ("china", "beijing"), ("russia", "moscow"), ("greece", "athens"), ("norway", "oslo"), ("cuba", "havana"),
             ("iran", "tehran"), ("germany", "berlin"), ("poland", "warsaw"), ("austria", "vienna"), ("sweden", "stockholm")]
animals = [("dog", "woof"), ("cat", "meow"), ("cow", "moo"), ("duck", "quack"), ("sheep", "baa"), ("owl", "hoot")]
templates = ["the capital of {c} is {k} .", "{k} is the capital of {c} .", "{k} is a big city in {c} .",
             "many people in {c} live in {k} ."]
corpus = [t.format(c=c, k=k) for c, k in countries for t in templates]
corpus += [f"the {a} says {s} ." for a, s in animals] + [f"a {a} can say {s} ." for a, s in animals]
corpus += ["the sky is blue .", "grass is green .", "snow is white .", "the sun is hot .", "ice is cold ."]

sft_pairs = ([(f"capital of {c} ?", f"{k} .") for c, k in countries]                        # short answers
             + [(f"where is {k} ?", f"{k} is in {c} .") for c, k in countries]                # medium
             + [(f"tell me about {k} .", f"{k} is the capital of {c} . many people in {c} live in {k} .")
                for c, k in countries[::2]]                                                  # long
             + [(f"what does a {a} say ?", f"a {a} says {s} .") for a, s in animals])

SPECIAL = ["<pad>", "<eot>", "<user>", "<assistant>", "<eos>"]
words = sorted({w for text in corpus + [q + " " + a for q, a in sft_pairs] for w in text.split()})
vocab = SPECIAL + words
stoi = {w: i for i, w in enumerate(vocab)}
PAD, EOT, USER, ASSISTANT, EOS = (stoi[s] for s in SPECIAL)
V = len(vocab)
encode = lambda text: [stoi[w] for w in text.split()]
decode = lambda ids: " ".join(vocab[i] for i in ids)

print(f"vocab: {V} tokens | pretraining corpus: {len(corpus)} sentences | SFT set: {len(sft_pairs)} Q&A pairs")
print("a pretraining document :", corpus[0], "->", encode(corpus[0]) + [EOT])
print("an SFT pair            :", sft_pairs[0])

The model is a standard decoder-only transformer: token + position embeddings, two pre-norm blocks with causal
self-attention, and a linear head over the vocabulary. The causal mask means position *t* only sees positions ≤ *t*,
so one forward pass yields a next-token prediction at **every** position — that is why next-token training is so
data-efficient (`tn02`).

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d=64, n_layer=2, n_head=4, max_len=32):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(max_len, d)
        block = nn.TransformerEncoderLayer(d, n_head, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, n_layer, enable_nested_tensor=False)
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)

    def forward(self, ids):                                   # ids: [batch, T] -> logits: [batch, T, vocab]
        T = ids.shape[1]
        x = self.tok(ids) + self.pos(torch.arange(T))
        causal = nn.Transformer.generate_square_subsequent_mask(T)
        return self.head(self.ln(self.blocks(x, mask=causal, is_causal=True)))

print(f"TinyGPT parameters: {sum(p.numel() for p in TinyGPT(V).parameters()):,}")

### The loss, including the `-100` convention

The training loss is cross-entropy between the prediction at position *t* and the actual token at *t + 1*: the
logits are shifted left by one against the labels. Any label equal to **-100** is skipped — PyTorch's
`cross_entropy(ignore_index=-100)` and every Hugging Face model's `forward(labels=...)` use this exact convention.
The mean is taken over the **non-ignored** positions only.

In [ ]:
def lm_loss(model, ids, labels):
    logits = model(ids)
    return F.cross_entropy(logits[:, :-1].reshape(-1, V), labels[:, 1:].reshape(-1), ignore_index=-100)

def pad_batch(seqs, labs):
    """Right-pad to the longest sequence. Pad positions get label -100."""
    T = max(map(len, seqs))
    ids = torch.full((len(seqs), T), PAD)
    lab = torch.full((len(seqs), T), -100)
    for i, (s, l) in enumerate(zip(seqs, labs)):
        ids[i, :len(s)] = torch.tensor(s)
        lab[i, :len(l)] = torch.tensor(l)
    return ids, lab

# check the convention by hand on random numbers: masked mean == ignore_index mean
logits, target = torch.randn(6, V), torch.tensor([3, 7, -100, 9, -100, 1])
per_token = F.cross_entropy(logits, target.clamp(min=0), reduction="none")
by_hand = per_token[target != -100].mean()
builtin = F.cross_entropy(logits, target, ignore_index=-100)
print(f"by hand: {by_hand:.4f}   ignore_index: {builtin:.4f}")
assert torch.allclose(by_hand, builtin)

Right padding is safe with a causal mask: every real token sits *before* the pads, so it can never attend to them,
and the pads' labels are `-100`, so they contribute nothing to the loss.

## 2. Pretrain briefly on plain text

Each document is its sentence followed by `<eot>`, and every token is a label — plain next-token prediction.

In [ ]:
def train(model, seqs, labs, steps, lr, batch_size=16, seed=0, eval_every=None, eval_fn=None):
    """Minimal training loop: sample a batch, forward, loss, backward, step. Returns a history of dicts."""
    pick = random.Random(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    history = []
    for step in range(1, steps + 1):
        idx = [pick.randrange(len(seqs)) for _ in range(batch_size)]
        ids, lab = pad_batch([seqs[i] for i in idx], [labs[i] for i in idx])
        loss = lm_loss(model, ids, lab)
        opt.zero_grad(); loss.backward(); opt.step()
        if eval_every and (step % eval_every == 0 or step == 1):
            history.append({"step": step, "train_loss": loss.item(), **(eval_fn(model) if eval_fn else {})})
    return history

pre_seqs = [encode(s) + [EOT] for s in corpus]
torch.manual_seed(0)
base = TinyGPT(V)
t0 = time.time()
pre_hist = train(base, pre_seqs, pre_seqs, steps=150, lr=5e-3, batch_size=32, eval_every=10)
print(f"pretraining: 150 steps in {time.time() - t0:.1f}s, final batch loss {pre_hist[-1]['train_loss']:.3f}")

plt.figure(figsize=(7, 3.2))
plt.plot([h["step"] for h in pre_hist], [h["train_loss"] for h in pre_hist])
plt.xlabel("step"); plt.ylabel("cross-entropy (nats/token)"); plt.yscale("log")
plt.title("Pretraining loss: next-token prediction on plain text"); plt.show()

The loss cannot reach zero: after "the capital of" any of 20 countries may follow, so the best possible prediction
there is a 1-in-20 guess. What *is* learnable — "the capital of france is" → "paris" — the model now knows. Let us
check, and also see what the **base model** does with a chat-formatted question.

In [ ]:
@torch.no_grad()
def generate(model, prompt_ids, max_new=12, stop=EOS):
    """Greedy decoding: append the most likely token until `stop` is produced or max_new is reached."""
    ids = list(prompt_ids)
    for _ in range(max_new):
        nxt = model(torch.tensor([ids]))[0, -1].argmax().item()
        ids.append(nxt)
        if nxt == stop:
            break
    return ids[len(prompt_ids):]

chat_prompt = lambda q: [USER] + encode(q) + [ASSISTANT]
base.eval()
print("completion :", "the capital of france is ->", decode(generate(base, encode("the capital of france is"), 3, stop=EOT)))
print("completion :", "the cow says ->", decode(generate(base, encode("the cow says"), 3, stop=EOT)))
for q in ["capital of france ?", "what does a cow say ?"]:
    print("chat       :", q, "->", decode(generate(base, chat_prompt(q), 8)))
known = sum(decode(generate(base, encode(f"the capital of {c} is"), 1)) == k for c, k in countries)
print(f"base model completes 'the capital of X is' correctly for {known}/{len(countries)} countries")
assert known == len(countries)

That is the base-vs-instruct distinction (`fo09`) in miniature. The **base model** has the knowledge — it completes
"the capital of france is" with "paris" — but it has never seen `<user>` / `<assistant>`, so a chat-formatted
question gets text-shaped noise, and it never emits `<eos>`. SFT fixes the *format*; the knowledge is already there.

## 3. Build SFT examples: `input_ids` and `labels`

An SFT example is `<user> question <assistant> answer <eos>`. The **labels** are a copy of `input_ids` with every
prompt position set to `-100`. Only the answer tokens — **including `<eos>`** — are learned.

In [ ]:
def build_example(question, answer, mask_prompt=True, eos_in_loss=True):
    prompt = [USER] + encode(question) + [ASSISTANT]
    completion = encode(answer) + [EOS]
    input_ids = prompt + completion
    labels = ([-100] * len(prompt) if mask_prompt else prompt.copy()) + completion.copy()
    if not eos_in_loss:
        labels[-1] = -100          # what a collator does if it masks "all pad tokens" and pad == eos
    return input_ids, labels

ids, labels = build_example(*sft_pairs[20])
print(f"{'position':>8}  {'input token':>12}  {'label':>6}  learned?")
for t, (i, l) in enumerate(zip(ids, labels)):
    print(f"{t:>8}  {vocab[i]:>12}  {l:>6}  {'yes -> ' + vocab[l] if l != -100 else ''}")

Remember the shift: the label at position *t* is the token the model must predict **from** position *t − 1*. So
`<assistant>` (the last prompt token, label -100) is where the model predicts the first answer word, and the last
answer word is where it predicts `<eos>`.

Here is the mask for a whole padded batch — the picture to have in your head when someone says "completion-only
loss":

In [ ]:
def show_mask(pairs, title, **kw):
    rows = [build_example(q, a, **kw) for q, a in pairs]
    T = max(len(r[0]) for r in rows)
    cat = np.zeros((len(rows), T))                     # 0 pad, 1 ignored (-100), 2 learned, 3 learned <eos>
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(rows) + 1))
    for r, (ids_, labs_) in enumerate(rows):
        for t, (i, l) in enumerate(zip(ids_, labs_)):
            cat[r, t] = 1 if l == -100 else (3 if i == EOS else 2)
            ax.text(t, r, vocab[i].replace("<assistant>", "<asst>"), ha="center", va="center", fontsize=7)
    ax.imshow(cat, cmap=ListedColormap(["white", "#dee2e6", "#8ce99a", "#ffa94d"]), vmin=0, vmax=3, aspect="auto")
    ax.set_xticks(range(T)); ax.set_yticks(range(len(rows))); ax.set_xlabel("position")
    ax.set_title(title + "   grey = label -100 (ignored), green = learned, orange = learned <eos>, white = pad")
    plt.show()

batch_pairs = [sft_pairs[i] for i in (0, 20, 40, 50, 33)]
show_mask(batch_pairs, "Completion-only (masked) labels.")
show_mask(batch_pairs, "Unmasked labels.", mask_prompt=False)

## 4. Train it: masked vs unmasked (and pretrained vs from scratch)

Same pretrained starting point, same data, same seed, same 80 steps — only the labels differ. We also fine-tune a
randomly initialised model with the masked labels, to see what pretraining contributed.

The metric is **exact match**: the fraction of SFT questions for which the model's greedy answer is exactly right,
`<eos>` included. We compute it with teacher forcing (one forward pass): if the argmax at every answer position
equals the label, greedy decoding would produce exactly that answer.

In [ ]:
EVAL_IDS, EVAL_LABELS = pad_batch(*zip(*[build_example(q, a) for q, a in sft_pairs]))   # answer-only labels

def metrics(model):
    """Exact match (teacher-forced) and answer-token loss on the SFT questions, from one forward pass."""
    with torch.no_grad():
        logits = model(EVAL_IDS)
    pred, target = logits.argmax(-1)[:, :-1], EVAL_LABELS[:, 1:]
    em = ((pred == target) | (target == -100)).all(dim=1).float().mean().item()
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, V), target.reshape(-1), ignore_index=-100).item()
    return {"exact_match": em, "answer_loss": loss}

def sft_run(start_model, steps=80, lr=3e-3, **label_kw):
    model = copy.deepcopy(start_model)
    rows = [build_example(q, a, **label_kw) for q, a in sft_pairs]
    hist = train(model, [r[0] for r in rows], [r[1] for r in rows], steps, lr, batch_size=16, seed=1,
                 eval_every=10, eval_fn=metrics)
    return model.eval(), hist

t0 = time.time()
masked, h_masked = sft_run(base)
unmasked, h_unmasked = sft_run(base, mask_prompt=False)
torch.manual_seed(0)
scratch, h_scratch = sft_run(TinyGPT(V))
print(f"3 SFT runs in {time.time() - t0:.1f}s")
em_at = lambda h, s: next(x["exact_match"] for x in h if x["step"] == s)
for name, h in [("pretrained + masked", h_masked), ("pretrained + unmasked", h_unmasked), ("scratch + masked", h_scratch)]:
    print(f"{name:22s} exact match: step 30 = {em_at(h, 30):.2f}, step 50 = {em_at(h, 50):.2f}, step 80 = {em_at(h, 80):.2f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for name, h, c in [("pretrained + masked", h_masked, "#2f9e44"), ("pretrained + unmasked", h_unmasked, "#e8590c"),
                   ("from scratch + masked", h_scratch, "#868e96")]:
    steps = [x["step"] for x in h]
    a1.plot(steps, [x["exact_match"] for x in h], label=name, color=c)
    a2.plot(steps, [x["train_loss"] for x in h], label=name + " (loss it optimises)", color=c, ls="--")
    a2.plot(steps, [x["answer_loss"] for x in h], label=name + " (answer tokens)", color=c)
a1.set_xlabel("SFT step"); a1.set_ylabel("exact match on SFT questions"); a1.set_title("Learning the answers"); a1.legend()
a2.set_xlabel("SFT step"); a2.set_ylabel("cross-entropy"); a2.set_yscale("log"); a2.set_title("Two different losses")
a2.legend(fontsize=7); plt.tight_layout(); plt.show()
assert h_masked[-1]["exact_match"] >= h_unmasked[-1]["exact_match"]
assert em_at(h_masked, 50) > 2 * em_at(h_scratch, 50)

**Read the charts.**

- **Pretrained vs scratch:** starting from the pretrained model, SFT learns the answers far faster — the facts were
  already in the weights, SFT only had to teach the format. From scratch, every fact has to be memorised from the
  56 examples; it catches up only late, by brute memorisation, which does not work on a real dataset. That is
  transfer learning, and it is why nobody runs SFT on a random network.
- **Masked vs unmasked:** with the same number of steps, the masked run gets at least as far on the answers. In the
  unmasked run most label positions are *question* tokens, so most of the gradient is spent learning to predict
  questions.
- **The dashed lines are not comparable.** The unmasked run's own loss includes question tokens, and those are
  partly unpredictable (which of 20 countries comes next?), so it plateaus far above the answer loss. Comparing
  "loss" across runs with different masking — or different templates — compares different things. Always evaluate
  every run on the *same* answer-only metric.

Let us check the teacher-forcing shortcut against real greedy decoding, then look at what else the unmasked model
learned.

In [ ]:
greedy_hits = [generate(masked, chat_prompt(q), max_new=20) == encode(a) + [EOS] for q, a in sft_pairs]
teacher_forced = metrics(masked)["exact_match"]
print(f"greedy exact match: {np.mean(greedy_hits):.2f}   teacher-forced exact match: {teacher_forced:.2f}")
assert np.isclose(np.mean(greedy_hits), teacher_forced)

print("\nWhat does each model write after a bare <user> token?")
probe = {name: generate(m, [USER], 9) for name, m in [("masked", masked), ("unmasked", unmasked)]}
for name, out in probe.items():
    print(f"  {name:9s} -> {decode(out)}")
assert ASSISTANT in probe["unmasked"] and ASSISTANT not in probe["masked"]

The unmasked model has learned to **write the user's side**: a plausible question, then `<assistant>`, then an
answer to its own question — because it was trained to predict questions. In a real assistant that shows up as a
model that continues the conversation on the user's behalf, or drifts toward the style of your prompts. The masked
model never learned the question distribution at all: its capacity went to answers.

> Nuance worth knowing: masking the prompt is the default in TRL and most recipes, but it is not a law of nature.
> Some studies (e.g. "Instruction Tuning With Loss Over Instructions", 2024) found that a small prompt loss can help
> when answers are very short and data is scarce. Treat it as a hyperparameter you can measure, with masking as the
> default.

## 5. The EOS token: how a model learns to stop

**In plain English:** a model does not "decide" an answer is finished. It emits a special end-of-turn token, and
the serving code stops when it sees it. If `<eos>` was never in the loss, the model never learned to produce it.

A classic way to get this wrong: many tokenizers set `pad_token = eos_token` (SmolLM2 does; so do many Llama recipes).
A collator that sets `labels[input_ids == pad_id] = -100` then masks every **real** EOS too. We simulate that bug
with `eos_in_loss=False`:

In [ ]:
no_eos, _ = sft_run(base, eos_in_loss=False)

probe_pairs = sft_pairs[::3]                      # every third question: all four kinds, fewer decode steps

def stop_report(model, n=18):
    outs = [generate(model, chat_prompt(q), max_new=n) for q, _ in probe_pairs]
    return np.mean([o[-1] == EOS for o in outs]), outs

rate_ok, outs_ok = stop_report(masked)
rate_bug, outs_bug = stop_report(no_eos)
for q_i in (0, 7, 17):
    print(f"Q: {probe_pairs[q_i][0]}")
    print(f"   eos in loss : {decode(outs_ok[q_i])}")
    print(f"   eos masked  : {decode(outs_bug[q_i])}")
print(f"\nstopped by itself: eos in loss {rate_ok:.0%} | eos masked {rate_bug:.0%}")
assert rate_ok == 1.0 and rate_bug < 0.5

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.bar(["<eos> in the loss", "<eos> masked (pad == eos bug)"], [rate_ok, rate_bug], color=["#2f9e44", "#e03131"])
plt.ylabel("answers that ended with <eos>"); plt.ylim(0, 1.05)
plt.title("A model only stops if it was trained to"); plt.show()

Without `<eos>` in the loss the answer is often right — and then it keeps going until `max_new_tokens`, falling
back on pretraining habits (`<eot>`, more sentences). In production this looks like rambling, runaway token bills and
answers that "continue the conversation". Two fixes: mask padding **by position** (attention mask / sequence length),
not by token id; or give the tokenizer a pad token that differs from EOS.

## 6. Packing vs padding

**In plain English:** a batch is a rectangle. Padding fills short rows with blanks — wasted compute. **Packing**
glues several examples end to end into each row, so almost nothing is wasted. The catch: examples in the same row
must not attend to each other.

Measure the waste on our SFT set:

In [ ]:
seqs = [build_example(q, a)[0] for q, a in sft_pairs]
lengths = np.array([len(s) for s in seqs])
BATCH, BLOCK = 8, 32

def pad_waste(order):
    real = padded = 0
    for i in range(0, len(order), BATCH):
        chunk = lengths[order[i:i + BATCH]]
        real += chunk.sum(); padded += chunk.max() * len(chunk)
    return 1 - real / padded

def pack(lengths_, block):
    """First-fit decreasing: place whole examples into rows of `block` tokens (no example is split)."""
    rows_ = []
    for i in sorted(range(len(lengths_)), key=lambda i: -lengths_[i]):
        for r in rows_:
            if sum(lengths_[j] for j in r) + lengths_[i] <= block:
                r.append(i); break
        else:
            rows_.append([i])
    return rows_

rng = np.random.default_rng(0)
packed_rows = pack(lengths, BLOCK)
waste = {"pad, random order": pad_waste(rng.permutation(len(lengths))),
         "pad, sorted by length": pad_waste(np.argsort(lengths)),
         f"pack into {BLOCK}-token rows": 1 - lengths.sum() / (len(packed_rows) * BLOCK)}
print(f"example lengths: min {lengths.min()}, median {int(np.median(lengths))}, max {lengths.max()}")
for k, v in waste.items():
    print(f"{k:24s}: {v:5.1%} of computed positions are padding")
print(f"packing: {len(sft_pairs)} examples -> {len(packed_rows)} rows of {BLOCK}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1]})
grid = np.zeros((BATCH, lengths[:BATCH].max()))
for r in range(BATCH):
    grid[r, :lengths[r]] = 1 + r % 2
a1.imshow(grid, cmap=ListedColormap(["white", "#74c0fc", "#4dabf7"]), aspect="auto")
a1.set_title(f"Padded batch: {1 - lengths[:BATCH].sum() / grid.size:.0%} white = wasted"); a1.set_xlabel("position")
pgrid = np.zeros((len(packed_rows[:BATCH]), BLOCK))
for r, members in enumerate(packed_rows[:BATCH]):
    pos = 0
    for k, i in enumerate(members):
        pgrid[r, pos:pos + lengths[i]] = 1 + k % 2; pos += lengths[i]
a2.imshow(pgrid, cmap=ListedColormap(["white", "#74c0fc", "#1c7ed6"]), aspect="auto")
a2.set_title("Packed rows: colours = different examples"); a2.set_xlabel("position")
plt.tight_layout(); plt.show()

Packing removes the waste, but a plain causal mask would let the second example in a row attend to the first — its
answer would be conditioned on someone else's conversation. Correct packing uses a **block-diagonal causal mask** and
**resets `position_ids`** at each example boundary:

In [ ]:
members = packed_rows[0]
seg = np.concatenate([[k] * lengths[i] for k, i in enumerate(members)])
T = len(seg)
causal = np.tril(np.ones((T, T)))
block_diag = causal * (seg[:, None] == seg[None, :])
position_ids = np.concatenate([np.arange(lengths[i]) for i in members])
print("examples in row 0:", [decode(seqs[i]) for i in members])
print("position_ids     :", position_ids.tolist())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, m, t in [(axes[0], causal, "plain causal mask: leaks across examples"),
                 (axes[1], block_diag, "block-diagonal causal mask: correct")]:
    ax.imshow(m, cmap="Blues"); ax.set_title(t, fontsize=10)
    ax.set_xlabel("attends to position"); ax.set_ylabel("query position")
plt.tight_layout(); plt.show()
assert block_diag.sum() == sum(lengths[i] * (lengths[i] + 1) / 2 for i in members)

In the Hugging Face stack this is `DataCollatorWithFlattening` (in `transformers`): it concatenates a batch into one
row, returns `position_ids` that restart at 0 for each example, and sets the label of each example's first token to
`-100`, so no loss crosses a boundary. Attention kernels that understand variable-length sequences (FlashAttention's
"varlen" path) use those boundaries to apply exactly the block-diagonal mask above, with no padding at all. Plain
"concatenate and chop into blocks" packing — common for *pretraining*, where documents are separated by `<eot>` —
accepts some cross-document attention in exchange for simplicity.

## 7. What SFT libraries do for you

Everything above — template, label mask, EOS, packing, the loop — is what TRL's `SFTTrainer` automates. TRL is not
installed here, and its argument names have changed between releases, so treat this as a sketch to check against
the docs of your version (names as of TRL 0.2x, 2025):

```python
from trl import SFTConfig, SFTTrainer

config = SFTConfig(
    output_dir="out",
    assistant_only_loss=True,   # conversational data: loss on assistant turns only
                                # (needs a chat template with {% generation %} markers, see notebook 01)
    completion_only_loss=True,  # prompt/completion data: loss on the completion only (the default there)
    packing=True,               # pack examples into full rows (with a padding-free / boundary-aware mask)
    max_length=1024,
    learning_rate=2e-4, num_train_epochs=2, per_device_train_batch_size=8, gradient_accumulation_steps=4,
)
trainer = SFTTrainer(model="HuggingFaceTB/SmolLM2-135M-Instruct", args=config, train_dataset=dataset)
trainer.train()
```

Older tutorials use `DataCollatorForCompletionOnlyLM(response_template=...)` for the same masking; recent TRL
versions replaced it with the config flags above. Whatever the library, verify the mask yourself: print one batch's
`labels` and check that only assistant tokens — and the EOS — are not `-100`. That one print catches the bugs in this
notebook.

## 8. Hyperparameters that matter

| Knob | Typical SFT value | What goes wrong |
|---|---|---|
| learning rate | full FT 1e-5 – 5e-5; LoRA 1e-4 – 3e-4 (roughly 10× higher, `tn14`) | too high wrecks the base (forgetting); too low: nothing moves |
| epochs | 1–3 | more usually memorises: train loss falls, eval loss rises (`tn28`) |
| effective batch size | 16–128 sequences (or tokens per step) via gradient accumulation | micro-batch 1 with no accumulation is noisy and unstable |
| schedule | linear warm-up (~3–10% of steps) then cosine or linear decay | no warm-up + high LR: early loss spikes |
| max length | cover ~99% of examples; drop or fix the rest | silent truncation cuts off answers *and their EOS* |
| loss mask / EOS / template | assistant-only, EOS inside the loss, same template as inference | the silent bugs from sections 3–5 |

Vocabulary check (`tn22`): a **batch** is the examples processed together, a **step** is one optimizer update, an
**epoch** is one pass over the dataset. Our SFT run: 56 examples, batch 16 → 3.5 steps per epoch, so 80 steps
≈ 23 epochs — fine for a toy that must memorise 56 facts, far too many for a real dataset.

A quick learning-rate sweep on our toy shows the classic shape:

In [ ]:
lrs = [1e-4, 1e-3, 3e-3, 1e-2, 5e-2]
t0 = time.time()
sweep = {lr: sft_run(base, steps=40, lr=lr)[1][-1] for lr in lrs}
print(f"sweep: {len(lrs)} runs x 40 steps in {time.time() - t0:.1f}s")
for lr, h in sweep.items():
    print(f"lr {lr:7.0e}: exact match {h['exact_match']:.2f}, answer loss {h['answer_loss']:.3f}")

plt.figure(figsize=(7, 3.2))
plt.semilogx(lrs, [sweep[lr]["answer_loss"] for lr in lrs], "o-")
plt.xlabel("learning rate (log)"); plt.ylabel("answer loss after 40 steps"); plt.yscale("log")
plt.title("Learning-rate sweep with a fixed step budget"); plt.show()

With a fixed step budget the smallest learning rates barely move the model, and past some point bigger steps stop
helping and start to hurt. The best value for a toy says nothing about your model — the lesson is the *shape*, and
that the sweep costs minutes on a toy and days on a real run, so start from the published ranges.

## Try it yourself

**1.** Build the SFT labels for a **two-turn** conversation (`<user> q1 <assistant> a1 <eos> <user> q2 <assistant>
a2 <eos>`) so that both answers (and both `<eos>`) are learned and both questions are masked. Print the learned
tokens.

In [ ]:
# Solution — try it yourself first, then run this
def build_multiturn(turns):
    ids_, labs_ = [], []
    for q, a in turns:
        prompt = [USER] + encode(q) + [ASSISTANT]
        completion = encode(a) + [EOS]
        ids_ += prompt + completion
        labs_ += [-100] * len(prompt) + completion
    return ids_, labs_

ids_, labs_ = build_multiturn([("what is the capital of peru ?", "lima ."), ("where is oslo ?", "oslo is in norway .")])
print("input :", decode(ids_))
print("learn :", decode([l for l in labs_ if l != -100]))
assert labs_.count(EOS) == 2

**2.** The "pad == eos" bug, reproduced the way it happens in real code: pad a batch with `EOS` as the pad token and
build labels with `labels[input_ids == pad_id] = -100`. How many real `<eos>` labels survive? Then fix it by masking
by position.

In [ ]:
# Solution — try it yourself first, then run this
rows = [build_example(q, a) for q, a in sft_pairs[:8]]
T = max(len(r[0]) for r in rows)
ids_ = torch.tensor([r[0] + [EOS] * (T - len(r[0])) for r in rows])          # pad token == EOS
lengths_ = torch.tensor([len(r[0]) for r in rows])
buggy = ids_.clone(); buggy[ids_ == EOS] = -100                              # masks pads AND the real EOS
fixed = ids_.clone(); fixed[torch.arange(T)[None, :] >= lengths_[:, None]] = -100   # mask by position
print(f"real <eos> labels kept - buggy: {(buggy == EOS).sum().item()} of 8, fixed: {(fixed == EOS).sum().item()} of 8")

**3.** Change the packing block size to 16, 24, 48 and 64 tokens. How does the padding waste change, and why does it
not keep falling for large blocks with this dataset?

In [ ]:
# Solution — try it yourself first, then run this
for block in (24, 32, 48, 64, 96):
    rows_ = pack(lengths, block)
    print(f"block {block:2d}: {len(rows_):2d} rows, waste {1 - lengths.sum() / (len(rows_) * block):5.1%}")

## Say it in an interview

- **Loss masking in one sentence:** "Labels are a copy of the input ids with the prompt positions set to -100, so
  cross-entropy — after the one-token shift — averages only over the assistant's tokens, EOS included."
- **Why mask:** otherwise the model learns to write the user's turn and spends its gradient on predicting prompts;
  and a loss that includes prompt tokens is not comparable with one that does not. (Masking is the default, not a
  law — a small prompt loss sometimes helps with tiny, short-answer datasets.)
- **EOS:** the model stops only if the end-of-turn token is in the loss. `pad_token == eos_token` plus "mask all pad
  ids" silently removes it — mask by position instead.
- **Packing:** fills rows with several examples to cut padding waste; must reset `position_ids` and use a
  block-diagonal mask (or accept cross-document attention in pretraining-style packing).
- **Debug order for "the fine-tune is worse than the base" (`tn32`):** chat template (same at train and inference?),
  loss mask (print one batch's labels), EOS in the loss, then data (read 50 rows), then LR/epochs, then the
  comparison itself (same prompts, decoding and template on both sides).
- **Base vs instruct:** the base model has the knowledge but not the format; SFT teaches the format — which is why SFT
  from a pretrained model is fast and SFT from scratch is not.

## Next

- [03 · LoRA from scratch](03_lora_from_scratch.ipynb) — train a fraction of the weights instead of all of them
- [04 · LoRA with PEFT](04_lora_with_peft.ipynb) — the same masking on a real chat model (SmolLM2) with its template
- [01 · When to fine-tune and data prep](01_when_to_fine_tune_and_data_prep.ipynb) — templates and data quality
- [Trainer API: fine-tune a small model](../10_huggingface/05_trainer_api_fine_tune_small_model.ipynb) · [Training loops and optimizers](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)